# Generative retrieval with semantic IDs (TIGER) on Amazon Reviews 2023

Runtime: **Runtime > Change runtime type > T4 GPU**, then run the cells one by one (cells 1 and 4 ask for a file / Drive access).

One category per run: set `CATEGORY` in cell 2 (`beauty`, `sports` or `toys`). Result: a model version folder saved to Google Drive and downloaded as `artifacts_<category>.zip`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Upload the code (`tiger-semantic-ids-code.zip`)

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()                     # choose tiger-semantic-ids-code.zip
zipfile.ZipFile(next(iter(up))).extractall('/content/tiger')
os.chdir('/content/tiger')
print(sorted(os.listdir('.')))

## 2. Download the category (5-core ratings + item metadata from the McAuley Lab server; no key needed)

The metadata file is large (1-2 GB for Beauty); it is streamed and only the sampled items are kept.

In [ ]:
CATEGORY = 'beauty'                     # beauty | sports | toys
NAME = {'beauty': 'Beauty_and_Personal_Care', 'sports': 'Sports_and_Outdoors', 'toys': 'Toys_and_Games'}[CATEGORY]
os.makedirs('/content/data', exist_ok=True)
for host in ('https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023', 'https://datarepo.eng.ucsd.edu/mcauley_group/data/amazon_2023'):
    r = os.system(f'wget -q -nc -P /content/data {host}/benchmark/5core/rating_only/{NAME}.csv.gz && '
                  f'wget -q -nc -P /content/data {host}/raw/meta_categories/meta_{NAME}.jsonl.gz')
    if r == 0:
        break
!ls -la /content/data

## 3. Tests (synthetic data, about 1 minute)

In [ ]:
!pip -q install fastapi uvicorn httpx
!python -m pytest -q -p no:warnings

## 4. Save results to Google Drive (survives a disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = f'/content/drive/MyDrive/tiger-semantic-ids/artifacts_{CATEGORY}'
os.makedirs(OUT, exist_ok=True)

## 5. Train and evaluate on the FULL 5-core data (Beauty: about 1.5 hours on a T4)

All users; Sentence-T5 embeddings (cached on Drive) -> RQ-VAE semantic IDs -> TIGER seq2seq (4 epochs over all training prefixes); SASRec and popularity baselines; test on 20,000 random held-out users (the same for every model).

A checkpoint is saved to Drive after every epoch: if Colab disconnects, re-run cells 1, 2, 4 and this cell and it continues from the last finished epoch.

In [ ]:
!python -m tiger.train --ratings /content/data/{NAME}.csv.gz --meta /content/data/meta_{NAME}.jsonl.gz --category {CATEGORY} --users 0 --epochs 3 --sasrec-epochs 15 --eval-users 20000 --eval-every 1 --batch 1024 --lr 2e-3 --out "$OUT" --notes "full 5-core, Colab T4" 2>&1 | grep --line-buffered -v -E "Warning|it/s\]" | tee -a /content/train_log.txt

## 6. Results

In [ ]:
import json, pandas as pd
v = open(f'{OUT}/LATEST').read().strip()
m = json.load(open(f'{OUT}/{v}/metrics.json'))
print('version:', v, '| category:', m['category_name'])
rows = []
for k, r in m['results'].items():
    rows.append({'model': k, **{f'test {x}': r['test'][x] for x in ('recall@5', 'recall@10', 'ndcg@5', 'ndcg@10')},
                 'cold R@10': r['test_cold']['recall@10'], 'warm R@10': r['test_warm']['recall@10'],
                 'val R@10': r['val']['recall@10'], 'train min': r.get('train_minutes'),
                 'invalid IDs (no trie)': r.get('invalid_id_rate_unconstrained')})
display(pd.DataFrame(rows))
print('cold test users:', m['results']['tiger']['test_cold']['users'], 'of', m['results']['tiger']['test']['users'])
print('semantic IDs:', json.dumps(m['semantic_ids'], indent=1))
print('data:', m['data'])
print('rq-vae last:', m['rqvae_history'][-1])
print('tiger history:', m['history']['tiger'])

## 7. Download for the laptop

In [ ]:
import shutil
shutil.copy('/content/train_log.txt', f'{OUT}/{v}_train_log.txt')
shutil.make_archive(f'/content/artifacts_{CATEGORY}', 'zip', OUT)
print(round(os.path.getsize(f'/content/artifacts_{CATEGORY}.zip') / 1e6), 'MB')
files.download(f'/content/artifacts_{CATEGORY}.zip')
files.download(f'{OUT}/{v}/metrics.json')